# Indexing and Search with Lucene

Apache Lucene is the open-source Java library at the core of most production
search platforms, including Elasticsearch, OpenSearch, and Solr. It provides the
full toolkit of this chapter: analysis, inverted indexing, term weighting, and
BM25 ranking. This demo runs real Lucene 10 code in a Java kernel: we compare
analyzers, build an index over the book collection, run field-scoped and boosted
queries, and read a score breakdown with `explain`.

<div style="border-left: 4px solid #D97706; background: rgba(217, 119, 6, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#D97706; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Requires the Ganymede Java kernel</strong><br>
Unlike the other demos, this one runs on a Java kernel, not Python, because it uses real Apache Lucene. Install the Ganymede Java 21 kernel before you run it: see the "Java kernel" section of the <a href="README.md">README.md</a> in this folder. Then select <code>Ganymede 2.1.1 (Java 21)</code> as the notebook kernel.
</div>

**Learning goals:**
- See how an analyzer turns text into indexed terms, and why indexing and querying must use the same one
- Build a Lucene index of documents with typed fields (text, string, numeric)
- Run keyword, multi-field, boosted, and filtered queries ranked by BM25
- Read an `explain` breakdown that exposes every term in the BM25 formula
- Understand segments: why Lucene never edits a document in place

**Prerequisites:** ch04-02 (BM25 scoring), ch04-03 (in-database full-text search)

This notebook runs on the Ganymede Java kernel. The first cell uses the `%%pom`
magic to download Lucene 10 from Maven Central onto the kernel classpath.

In [1]:
%%pom
dependencies:
- org.apache.lucene:lucene-core:10.2.1
- org.apache.lucene:lucene-analysis-common:10.2.1
- org.apache.lucene:lucene-queryparser:10.2.1

## Analyzers

Tokenization is the job of an **analyzer**, and the choice of analyzer must be
identical for indexing and for querying. The standard analyzer lowercases and
strips punctuation; the English analyzer additionally removes stop words and
applies a Porter stemmer. Changing the analyzer means rebuilding the index.

In [2]:
import org.apache.lucene.analysis.Analyzer;
import org.apache.lucene.analysis.TokenStream;
import org.apache.lucene.analysis.en.EnglishAnalyzer;
import org.apache.lucene.analysis.standard.StandardAnalyzer;
import org.apache.lucene.analysis.tokenattributes.CharTermAttribute;
import java.util.*;

// Run one analyzer over a piece of text and collect the resulting terms
List<String> analyze(Analyzer analyzer, String text) throws Exception {
    List<String> terms = new ArrayList<>();
    TokenStream stream = analyzer.tokenStream("field", text);
    CharTermAttribute term = stream.addAttribute(CharTermAttribute.class);
    stream.reset();
    while (stream.incrementToken()) terms.add(term.toString());
    stream.end();
    stream.close();
    return terms;
}

String sample = "I think text's values' color goes here; WHAT happens";
System.out.println("Text:     " + sample);
System.out.println("Standard: " + analyze(new StandardAnalyzer(), sample));
System.out.println("English:  " + analyze(new EnglishAnalyzer(), sample));

Text:     I think text's values' color goes here; WHAT happens
Standard: [i, think, text's, values, color, goes, here, what, happens]
English:  [i, think, text, valu, color, goe, here, what, happen]


The standard analyzer keeps every word (lowercased, punctuation stripped). The
English analyzer removes stop words and stems, so "values" becomes "valu" and
"goes" becomes "goe". These stemmed forms are what actually go into the inverted
index, which is why a search for "going" can still match a document that says
"goes".

## Building an Index

Lucene models a document as a set of named **fields**, and each field has a type
that controls whether its value is tokenized, indexed, and stored. We index the
same book collection used in the other chapter 4 demos: `title` and `author` are
tokenized text fields, `genre` is an exact-match string field, and `year` is a
numeric field for range queries.

In [3]:
import org.apache.lucene.document.*;
import org.apache.lucene.index.*;
import org.apache.lucene.store.*;

// The 50-book collection, one book per line: id|title|author|genre|year
String data = """
database-systems|Database Systems: The Complete Book|Hector Garcia-Molina, Jeffrey D. Ullman, and Jennifer Widom|Computer Science|2008
pattern-recognition|Pattern Recognition and Machine Learning|Christopher Bishop|Computer Science|2006
data-science-scratch|Data Science from Scratch|Joel Grus|Computer Science|2015
operating-systems|Operating System Concepts|Abraham Silberschatz, Peter B. Galvin, and Greg Gagne|Computer Science|2018
ai-modern-approach|Artificial Intelligence: A Modern Approach|Stuart Russell and Peter Norvig|Computer Science|2020
programming-languages|Concepts of Programming Languages|Robert W. Sebesta|Computer Science|2015
theory-of-computation|Introduction to the Theory of Computation|Michael Sipser|Computer Science|2012
everyday-things|The Design of Everyday Things|Don Norman|Computer Science|2013
computer-organization|Computer Organization and Design: The Hardware/Software Interface|David A. Patterson and John L. Hennessy|Computer Science|2017
computer-networks|Computer Networks|Andrew Tanenbaum and David Wetherall|Computer Science|2010
cryptography|Cryptography and Network Security: Principles and Practice|William Stallings|Computer Science|2016
sicp|Structure and Interpretation of Computer Programs|Harold Abelson and Gerald Jay Sussman|Computer Science|1996
mythical-man-month|The Mythical Man-Month: Essays on Software Engineering|Frederick P. Brooks Jr.|Computer Science|1975
computer-systems|Computer Systems: A Programmer's Perspective|Randal E. Bryant and David R. O'Hallaron|Computer Science|2015
algorithms|Introduction to Algorithms|Thomas H. Cormen, Charles E. Leiserson, Ronald L. Rivest, and Clifford Stein|Computer Science|2022
blind-assassin|The Blind Assassin|Margaret Atwood|Fiction|2000
angels-in-america|Angels in America|Tony Kushner|Drama|1991
selected-poems|Selected Poems|Gwendolyn Brooks|Poetry|1963
handmaids-tale|The Handmaid's Tale|Margaret Atwood|Fiction|1985
penguin-history|The Penguin History of the World|J.M. Roberts|Non-Fiction|1980
ragtime|Ragtime|E.L. Doctorow|Fiction|1975
buried-child|Buried Child|Sam Shepard|Drama|1978
ariel|Ariel|Sylvia Plath|Poetry|1965
mockingbird|To Kill a Mockingbird|Harper Lee|Fiction|1960
new-architecture|Towards a New Architecture|Le Corbusier|Non-Fiction|1923
catcher-in-the-rye|The Catcher in the Rye|J.D. Salinger|Fiction|1951
death-of-a-salesman|Death of a Salesman|Arthur Miller|Drama|1949
on-the-road|On the Road|Jack Kerouac|Fiction|1957
brave-new-world|Brave New World|Aldous Huxley|Fiction|1932
room-of-ones-own|A Room of One's Own|Virginia Woolf|Non-Fiction|1929
great-gatsby|The Great Gatsby|F. Scott Fitzgerald|Fiction|1925
waste-land|The Waste Land|T.S. Eliot|Poetry|1922
swanns-way|Swann's Way|Marcel Proust|Fiction|1913
demonology|Letters on Demonology and Witchcraft|Walter Scott|Non-Fiction|1830
wind-in-the-willows|The Wind in the Willows|Kenneth Grahame|Fiction|1908
songs-of-innocence|Songs of Innocence and of Experience|William Blake|Poetry|1794
dracula|Dracula|Bram Stoker|Fiction|1897
interpretation-of-dreams|The Interpretation of Dreams|Sigmund Freud|Non-Fiction|1899
huckleberry-finn|Adventures of Huckleberry Finn|Mark Twain|Fiction|1884
dolls-house|A Doll's House|Henrik Ibsen|Drama|1879
flowers-of-evil|Flowers of Evil|Charles Baudelaire|Poetry|1857
crime-and-punishment|Crime and Punishment|Fyodor Dostoevsky|Fiction|1866
origin-of-species|On the Origin of Species|Charles Darwin|Non-Fiction|1859
madame-bovary|Madame Bovary|Gustave Flaubert|Fiction|1856
leaves-of-grass|Leaves of Grass|Walt Whitman|Poetry|1855
jane-eyre|Jane Eyre|Charlotte Bronte|Fiction|1847
frederick-douglass|Narrative of the Life of Frederick Douglass, an American Slave|Frederick Douglass|Non-Fiction|1845
macbeth|Macbeth|William Shakespeare|Drama|1623
prometheus-unbound|Prometheus Unbound|Percy Bysshe Shelley|Poetry|1820
frankenstein|Frankenstein|Mary Shelley|Fiction|1818
""";

// Build an in-memory index with the English analyzer
Directory directory = new ByteBuffersDirectory();
EnglishAnalyzer analyzer = new EnglishAnalyzer();
IndexWriter writer = new IndexWriter(directory, new IndexWriterConfig(analyzer));

int count = 0;
for (String line : data.strip().split("\n")) {
    String[] f = line.split("\\|");
    Document doc = new Document();
    doc.add(new StringField("doc_id", f[0], Field.Store.YES));  // exact match, stored
    doc.add(new TextField("title", f[1], Field.Store.YES));     // tokenized + stored
    doc.add(new TextField("author", f[2], Field.Store.YES));    // tokenized + stored
    doc.add(new StringField("genre", f[3], Field.Store.YES));   // exact match
    doc.add(new IntField("year", Integer.parseInt(f[4]), Field.Store.YES));  // numeric
    writer.addDocument(doc);
    count++;
}
writer.close();
System.out.println("Indexed " + count + " books into an in-memory Lucene index.");

Indexed 50 books into an in-memory Lucene index.


Sept 27, 2026 1:41:58 PM org.apache.lucene.internal.vectorization.VectorizationProvider lookup


Each `TextField` is tokenized and indexed with term frequencies and positions.
The `StringField` values are indexed verbatim for exact-match filtering, and the
`IntField` supports range queries on the publication year. Note that Lucene 10
requires an explicit `Field.Store` argument when constructing a numeric field.

## Querying and Scoring

A search uses an `IndexSearcher` over the index, usually with a query parser that
applies the same analyzer used at indexing time. Lucene ranks with BM25 by
default. We open a reader and run a multi-field query across title and author.

In [4]:
import org.apache.lucene.search.*;
import org.apache.lucene.queryparser.classic.MultiFieldQueryParser;
import org.apache.lucene.queryparser.classic.QueryParser;

IndexReader reader = DirectoryReader.open(directory);
IndexSearcher searcher = new IndexSearcher(reader);
StoredFields storedFields = searcher.storedFields();  // Lucene 10 API

// Print the top hits of a query as an aligned table
void showHits(Query query, int limit) throws Exception {
    TopDocs top = searcher.search(query, limit);
    System.out.printf("%-6s %-10s %-24s %s%n", "Rank", "BM25", "Doc ID", "Title");
    int rank = 1;
    for (ScoreDoc sd : top.scoreDocs) {
        Document d = storedFields.document(sd.doc);
        System.out.printf("%-6d %-10.4f %-24s %s%n",
            rank++, sd.score, d.get("doc_id"), d.get("title"));
    }
}

MultiFieldQueryParser parser =
    new MultiFieldQueryParser(new String[]{"title", "author"}, analyzer);
Query q = parser.parse("computer systems design");
System.out.println("Query: computer systems design (title + author)\n");
showHits(q, 10);

Query: computer systems design (title + author)

Rank   BM25       Doc ID                   Title
1      1.8419     computer-systems         Computer Systems: A Programmer's Perspective
2      1.5671     computer-organization    Computer Organization and Design: The Hardware/Software Interface
3      1.2983     everyday-things          The Design of Everyday Things
4      1.1534     operating-systems        Operating System Concepts
5      1.1238     computer-networks        Computer Networks
6      1.0058     database-systems         Database Systems: The Complete Book
7      0.9588     theory-of-computation    Introduction to the Theory of Computation
8      0.8361     sicp                     Structure and Interpretation of Computer Programs


Beyond plain keyword queries, Lucene lets us build queries programmatically,
combining clauses that **must**, **should**, or **must-not** match, or that only
**filter** without affecting the score. One catch: unlike the query parser, a
`TermQuery` does not run any analyzer. A common mistake is to take keywords
straight from a user interface and feed them into these clauses as-is: a raw
`new Term("title", "computer")` never matches, because the index holds the
analyzed form `comput`. The fix is to run each query word through the same
`EnglishAnalyzer` used at indexing time. The query below does that: it requires
`computer` (which the analyzer stems to `comput`) in the title, boosts titles that
also mention `design`, and filters to books published between 1990 and 2020.

In [5]:
import org.apache.lucene.index.Term;

// TermQuery does not analyze its term, so run each query word through the SAME
// analyzer used at indexing time (turns "computer" -> "comput", "design" -> "design").
String titleMust   = analyze(analyzer, "computer").get(0);
String titleShould = analyze(analyzer, "design").get(0);

Query filtered = new BooleanQuery.Builder()
    .add(IntField.newRangeQuery("year", 1990, 2020), BooleanClause.Occur.FILTER)
    .add(new TermQuery(new Term("title", titleMust)), BooleanClause.Occur.MUST)
    .add(new BoostQuery(new TermQuery(new Term("title", titleShould)), 2.0f),
         BooleanClause.Occur.SHOULD)
    .build();

System.out.println("MUST title:" + titleMust + ", SHOULD title:" + titleShould
    + "^2, FILTER year in [1990, 2020]\n");
showHits(filtered, 10);

MUST title:comput, SHOULD title:design^2, FILTER year in [1990, 2020]

Rank   BM25       Doc ID                   Title
1      2.4685     computer-organization    Computer Organization and Design: The Hardware/Software Interface
2      1.1238     computer-networks        Computer Networks
3      0.9588     theory-of-computation    Introduction to the Theory of Computation
4      0.8361     sicp                     Structure and Interpretation of Computer Programs
5      0.8361     computer-systems         Computer Systems: A Programmer's Perspective


The filter clause narrows the candidate set by year but contributes nothing to
the score. The required clause must match. The boosted optional clause pushes
documents that also mention "design" higher without excluding those that do not.

## Reading an `explain`

The `explain` method prints exactly how each clause contributed to a document's
score. Every quantity from the BM25 formula is visible: the IDF with its
`log(1 + ...)` variant, the saturating TF with `k1`, `b`, `dl`, and `avgdl`, and
any boost factor. This is the single most useful tool for understanding and
tuning relevance.

In [6]:
Query explainQuery = parser.parse("computer design");
TopDocs top = searcher.search(explainQuery, 1);
int topDocId = top.scoreDocs[0].doc;

System.out.println("Top hit: " + storedFields.document(topDocId).get("title"));
System.out.println("Query:   computer design\n");
System.out.println(searcher.explain(explainQuery, topDocId));

Top hit: Computer Organization and Design: The Hardware/Software Interface
Query:   computer design

1.5670851 = sum of:
  0.66570246 = weight(title:comput in 8) [BM25Similarity], result of:
    0.66570246 = score(freq=1.0), computed as boost * idf * tf from:
      2.2270775 = idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
        5 = n, number of documents containing term
        50 = N, total number of documents with field
      0.298913 = tf, computed as freq / (freq + k1 * (1 - b + b * dl / avgdl)) from:
        1.0 = freq, occurrences of term within document
        1.2 = k1, term saturation parameter
        0.75 = b, length normalization parameter
        6.0 = dl, length of field
        2.64 = avgdl, average length of field
  0.9013827 = weight(title:design in 8) [BM25Similarity], result of:
    0.9013827 = score(freq=1.0), computed as boost * idf * tf from:
      3.0155349 = idf, computed as log(1 + (N - n + 0.5) / (n + 0.5)) from:
        2 = n, number of document

Read the tree from the leaves up: each `weight(field:term)` node reports the IDF
(from `N` and `n`, the collection size and document frequency) and the TF (from
the raw frequency, `k1`, `b`, the field length `dl`, and the average `avgdl`).
The clause scores sum to the document's total. This is the same additive BM25 we
implemented by hand in ch04-02, now computed per field by a production engine.

Lucene applies one optimization worth noting: its `BM25Similarity` leaves out the
constant `(k1 + 1)` factor that the textbook BM25 (and our ch04-02 code) puts in
the numerator (it also uses the `log(1 + ...)` IDF variant). Because `(k1 + 1)` is
the same for every document, it only rescales the scores and cannot change the
ranking, so Lucene ignores it: its scores come out roughly half of ch04-02's at
the default `k1 = 1.2` while the order stays the same.

## Segments and Updates

Lucene writes the index as a set of immutable **segments**. Each flush creates a
new segment, and because segments are immutable Lucene never edits a document in
place: an update is a delete of the old version followed by an add of the new
one. Deletes identify documents by a term in a key field rather than by position.
We reopen the index in write mode, delete one book by its `doc_id`, and confirm
the live document count drops.

In [7]:
IndexWriter writer2 = new IndexWriter(directory, new IndexWriterConfig(analyzer));
System.out.println("Live documents before delete: " + writer2.getDocStats().numDocs);

// Delete by term: mark every document whose doc_id field equals this value
writer2.deleteDocuments(new Term("doc_id", "frankenstein"));
writer2.commit();
System.out.println("Live documents after delete:  " + writer2.getDocStats().numDocs);
writer2.close();

Live documents before delete: 50
Live documents after delete:  49


The delete flips a bit in the segment's live-documents list rather than erasing
data; the bytes are only reclaimed later when a merge rewrites the segment. This
segment model is what lets many readers search an index concurrently while writes
proceed, and it is the seed of the distributed scaling covered in the next
section of the chapter.

## When to Use Lucene

Lucene itself is a library you embed in a Java application. In practice most
teams use a server built on it, Elasticsearch, OpenSearch, or Solr, which add
clustering, replication, a REST API, and operational tooling. Reach for a
Lucene-based engine when search quality is critical and you need fine-grained
relevance tuning, when you need faceting, autocomplete, or highlighting, or when
your collection outgrows a single database. If search is only a feature of an
app you already back with a relational database, the in-database full-text search
from the previous demo is usually the lighter choice.

## Summary

| Concept | Key point |
|---------|-----------|
| Analyzer | Turns text into indexed terms; must match at index and query time |
| EnglishAnalyzer | Lowercases, removes stop words, applies a Porter stemmer |
| Field types | TextField (tokenized), StringField (exact), IntField (numeric, needs Field.Store) |
| IndexSearcher | Runs queries over a reader; ranks with BM25 by default |
| MultiFieldQueryParser | Parses user text into a query across several fields |
| Clause occur | MUST, SHOULD, MUST_NOT, and FILTER (matches but does not score) |
| BoostQuery | Multiplies a clause's contribution to favor some matches |
| explain | Prints IDF, TF, k1, b, dl, avgdl, and boost per clause |
| Segments | Immutable; updates are delete-plus-add, cleaned up on merge |

## Try It Yourself

1. Add a `genre` filter to the boosted query using
   `new TermQuery(new Term("genre", "Computer Science"))` with `Occur.FILTER`.
   How does the result set change?

2. Compare the ranking of `computer systems design` here with the same query in
   ch04-02 (hand-rolled BM25) and ch04-03 (SQLite FTS5). Do the top documents agree?

3. Run `explain` on the second-ranked document instead of the first. Which clause
   makes the difference between the two?

In [8]:
// Exercise: a genre-filtered, boosted query.
// genre is a non-analyzed StringField, so its term is matched verbatim ("Computer
// Science"); the title terms reuse the analyzer-derived forms from the query above.
Query exercise = new BooleanQuery.Builder()
    .add(new TermQuery(new Term("genre", "Computer Science")), BooleanClause.Occur.FILTER)
    .add(new TermQuery(new Term("title", titleMust)), BooleanClause.Occur.MUST)
    .add(new BoostQuery(new TermQuery(new Term("title", titleShould)), 2.0f),
         BooleanClause.Occur.SHOULD)
    .build();

System.out.println("FILTER genre:Computer Science, MUST title:comput, SHOULD title:design^2\n");
showHits(exercise, 10);

reader.close();

FILTER genre:Computer Science, MUST title:comput, SHOULD title:design^2

Rank   BM25       Doc ID                   Title
1      2.4685     computer-organization    Computer Organization and Design: The Hardware/Software Interface
2      1.1238     computer-networks        Computer Networks
3      0.9588     theory-of-computation    Introduction to the Theory of Computation
4      0.8361     sicp                     Structure and Interpretation of Computer Programs
5      0.8361     computer-systems         Computer Systems: A Programmer's Perspective
